# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
Dataset Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print summary info
metadata = dataset.metadata.to_json()
print(f"Name: {metadata.get('name','Unknown')}")
print(f"Description: {metadata.get('description','No description available.')}")
print(f"Published: {metadata.get('datePublished', 'N/A')}")
print(f"Version: {metadata.get('version', 'N/A')}")
print("Cite as: ", metadata.get('citeAs',''))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets (by @id) and their fields
print("Available record sets and their fields:\n")

record_sets = []
record_sets_info = []
for record_set in dataset.record_sets:
    print(f"  Record set: {record_set.id}")
    print(f"    Name: {getattr(record_set, 'name', '(unnamed)')}")
    fields = getattr(record_set, 'fields', [])
    if fields:
        print("    Fields:")
        for field in fields:
            print(f"      - {field.id}   (name: {getattr(field, 'name', '')}, dtype: {getattr(field, 'data_type', '')})")
    record_sets.append(record_set.id)
    record_sets_info.append({
        'id': record_set.id,
        'name': getattr(record_set, 'name', '(unnamed)'),
        'fields': [field.id for field in getattr(record_set, 'fields', [])]
    })
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For demonstration, we'll load each available record set into a separate DataFrame
# You can choose the desired record set by changing the variable below
dfs = {}
for record_set_id in record_sets:
    print(f"Loading records from record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    dfs[record_set_id] = pd.DataFrame(records)
    print(f"  -> {len(dfs[record_set_id])} records loaded.")

# For demonstration let's use the first record set
main_record_set_id = record_sets[0]
print(f"\nMain record set used: {main_record_set_id}")
print("Columns:", dfs[main_record_set_id].columns.tolist())
dfs[main_record_set_id].head(3)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Choose a numeric field for EDA automatically (the first numeric column found), else fallback to the first field
import numpy as np

df = dfs[main_record_set_id]
numeric_fields = []
for col in df.columns:
    try:
        # Convert to numeric, errors='coerce': NaN if not numeric
        values = pd.to_numeric(df[col], errors='coerce')
        if np.isfinite(values).sum() > 0:
            numeric_fields.append(col)
    except Exception:
        pass

if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    numeric_field = df.columns[0]

print(f"Numeric field selected for EDA: {numeric_field}\n")

# Filter rows where numeric_field > threshold (e.g. 10)
threshold = 10
values = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[values > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the field
if len(filtered_df) > 0:
    filtered_numeric = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    mean = filtered_numeric.mean()
    std = filtered_numeric.std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_numeric - mean) / std
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print(f"No records with {numeric_field} > {threshold}.")

# Attempt to find a group-by field (first categorical field that isn't the numeric one)
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].dtype == object and df[col].nunique() < max(20, len(df) // 10):
        group_field = col
        break

if group_field and (group_field in filtered_df.columns):
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
    print(f"\nGrouped data by {group_field} (showing mean of {numeric_field}):")
    print(grouped_df.head())
else:
    print("\nNo suitable categorical grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field if possible
plt.figure(figsize=(8, 4))
sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouping field exists, plot group-wise means
if group_field and group_field in df.columns:
    plt.figure(figsize=(8, 4))
    group_means = df.groupby(group_field)[numeric_field].mean().dropna().sort_values()
    group_means.plot(kind='bar')
    plt.title(f"Mean {numeric_field} grouped by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset. We reviewed the available record sets and their fields (all referenced using their Croissant `@id`s), extracted tabular data, performed filtering and normalization, and visualized the distribution of a key numeric field. This approach can be extended for further analysis, feature engineering, and downstream machine learning tasks as needed.